# Aggregate all training/eval JSONs

Scans `/content/drive/MyDrive/climate_data/` for every result file produced by any phase of the project (noncausal v4, V5, V5-mini, Path C+ Option C seed 42 v1/v2, Phase 6 dualpath, Phase 7 warm-start eval) and dumps them in a single text file the user can hand to the expert team.

Pure read-only — no training, no writing to anything except `/content/all_results_dump.txt`.

In [ ]:
# === Cell 1 : Mount Drive ===
import os, sys, json
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
OUT_PATH   = Path('/content/all_results_dump.txt')
print(f'DRIVE_ROOT = {DRIVE_ROOT} (exists={DRIVE_ROOT.exists()})')

In [ ]:
# === Cell 2 : Discover all JSON files ===
# Patterns we care about (any phase, any run dir).
PATTERNS = [
    'final_validation_metrics.json',
    'domain_metrics.json',
    'aligned_metrics_*.json',
    'training_history.json',
    'summary.json',
    'comparison_3way*.json',
    'comparison_3way_table.json',
    'seed42_vs_noncausal.json',
    'smoke_comparison.json',
    'smoke_partial.json',
    'eval_protocol_metrics.json',
    'eval_*.json',
    'metrics*.json',
    'ablation*.json',
    'dag*.json',
    'q_phys*.json',
    'pathcplus*.json',
    'oracle*.json',
    'phase*_metrics.json',
    'training_summary.json',
    'eval_summary.json',
    'shortcut_diagnostic.json',
]

# Directories to scan (top-level ckpt / run dirs).
CANDIDATE_DIRS = [
    DRIVE_ROOT,
    DRIVE_ROOT / 'ckpt_v2_corrdiff_normal',
    DRIVE_ROOT / 'ckpt_noncausal',
    DRIVE_ROOT / 'ckpt_phase6_dualpath',
    DRIVE_ROOT / 'ckpt_v5_mini',
    DRIVE_ROOT / 'ckpt_v5_pearson_090',
    DRIVE_ROOT / 'ckpt',
    DRIVE_ROOT / 'oracle_full',
    DRIVE_ROOT / 'oracle_9node',
    DRIVE_ROOT / 'oracle_9node' / 'seed_42',
    DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'phase7_smoke_compare',
    DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'phase7_stage2_retrain',
    DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'phase7_stage2_B_finetune',
    DRIVE_ROOT / 'oracle_9node' / 'seed_7',
    DRIVE_ROOT / 'oracle_9node' / 'seed_123',
]

found = []
for base in CANDIDATE_DIRS:
    if not base.exists():
        continue
    for pat in PATTERNS:
        for p in base.rglob(pat):
            if p.is_file() and p not in found:
                found.append(p)

found.sort()
print(f'Found {len(found)} JSON files :')
for p in found:
    print(f'  {p}')


In [ ]:
# === Cell 3 : Dump all JSON contents to one file + display tail ===
import io

buf = io.StringIO()
buf.write(f'# ALL RESULTS DUMP -- generated {datetime.now().isoformat()}\n')
buf.write(f'# DRIVE_ROOT = {DRIVE_ROOT}\n')
buf.write(f'# Total files = {len(found)}\n')
buf.write('\n')

n_ok = 0
n_err = 0
for p in found:
    rel = p.relative_to(DRIVE_ROOT) if str(p).startswith(str(DRIVE_ROOT)) else p
    buf.write('=' * 100 + '\n')
    buf.write(f'FILE : {rel}\n')
    buf.write(f'SIZE : {p.stat().st_size} bytes\n')
    buf.write(f'MTIME: {datetime.fromtimestamp(p.stat().st_mtime).isoformat()}\n')
    buf.write('=' * 100 + '\n')
    try:
        txt = p.read_text(encoding='utf-8', errors='replace')
        # If JSON, pretty-print it back to keep formatting consistent.
        try:
            data = json.loads(txt)
            txt = json.dumps(data, indent=2, default=str, ensure_ascii=False)
        except Exception:
            pass
        buf.write(txt)
        n_ok += 1
    except Exception as e:
        buf.write(f'[READ ERROR] {type(e).__name__}: {e}')
        n_err += 1
    buf.write('\n\n')

OUT_PATH.write_text(buf.getvalue(), encoding='utf-8')
print(f'WROTE : {OUT_PATH}  (size = {OUT_PATH.stat().st_size:,} bytes)')
print(f'  files read OK : {n_ok}')
print(f'  files errored : {n_err}')
print()
print('--- TAIL (last 2000 chars) ---')
print(buf.getvalue()[-2000:])

In [ ]:
# === Cell 4 : Copy to Drive + provide download link ===
import shutil

DRIVE_OUT = DRIVE_ROOT / 'all_results_dump.txt'
shutil.copy(str(OUT_PATH), str(DRIVE_OUT))
print(f'COPIED to Drive : {DRIVE_OUT}')

try:
    from google.colab import files
    files.download(str(OUT_PATH))
    print('Download triggered.')
except Exception as _e:
    print(f'[colab download skipped] {_e}')